# 03 — Pandas data pipelines


In [ ]:
from pathlib import Path

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'data' / 'output'
OUT.mkdir(parents=True, exist_ok=True)


🔴 Задача «Сквозной пайплайн: от сырых логов к метрикам»
Контекст: Ты получил два "грязных" CSV: users.csv (user_id, signup_date, region) и orders.csv (order_id, user_id, order_date, amount).

ТЗ:
### Загрузи оба файла с явным dtype и parse_dates. Приведи user_id к str в обоих.
### Объедини таблицы через left merge (все пользователи + их заказы). Проверь целостность ключей (validate).
### Отфильтруй заказы только за 2023 год. Заполни пропуски в amount нулями (пользователи без заказов).
### Посчитай для каждого region:
    ### total_revenue (сумма)
    ### avg_order (среднее)
    ### active_users (число уникальных user_id с amount > 0)
### Добавь колонку revenue_share: доля выручки региона от общей.
### Сохрани результат в metrics_by_region.csv без индекса. ✅

### Критерий успеха: Код <30 строк, 0 циклов, 0 предупреждений, файл открывается в Excel без кракозябр.

In [ ]:
import pandas as pd, numpy as np
users = pd.read_csv(str(RAW / 'users.csv'), dtype={'user_id': 'str', 'region': 'str'}).drop_duplicates(subset=['user_id'])
orders = pd.read_csv(str(RAW / 'orders.csv'), dtype={'order_id': 'str', 'user_id': 'str', 'amount': 'float32'})
users['signup_date'] = pd.to_datetime(users['signup_date'], format='%d.%m.%Y')
orders['order_date'] = pd.to_datetime(orders['order_date'], format='%d.%m.%Y')

merged = pd.merge(users, orders, on='user_id', how='left', validate='1:m')
merged = merged[merged['order_date'].dt.year == 2023]
merged['amount'] = merged['amount'].fillna(0)

df = merged[merged['amount'] > 0].groupby('region').agg(total_revenue=('amount','sum'), avg_order=('amount','mean'), active_users=('user_id','nunique')).copy()

df['revenue_share'] = df['total_revenue'] / (df['total_revenue'].sum())

df.to_csv(str(OUT / 'metrics_by_region.csv'), index=False)


🔴 Задача «Высокодоходные кварталы» (sales_wide)
Контекст: sales_wide.csv — выручка по product, region и кварталам (широкий формат).

ТЗ:
# Загрузи sales_wide.csv, преобразуй в long-формат через melt. Якорь: ['product', 'region']. Новые колонки: quarter, revenue.
# С помощью transform('sum') добавь колонку total_year (сумма выручки по группе product+region).
# Отфильтруй строки, где revenue > 0.35 * total_year (квартал дал >35% годовой выручки данной пары).
# Разверни отфильтрованные данные через pivot_table: index='product', columns='quarter', values='revenue', aggfunc='sum'. Заполни пропуски нулями.
# Экспортируй в high_impact_quarters.csv без индекса.

In [ ]:
import pandas as pd, numpy as np
transactions = pd.read_csv(str(RAW / 'transactions.csv'), dtype={'user_id': 'str', 'amount': 'float32'}, parse_dates=['transaction_date'])
# U0078,2023-01-01 00:00:00,1125.81

sales_wide = pd.read_csv(str(RAW / 'sales_wide.csv'))
# Widget_A,North,9980,8451,9906,1775
df = sales_wide.melt(id_vars=['product','region'], value_vars=['q1_rev', 'q2_rev', 'q3_rev', 'q4_rev'],var_name = 'quarter', value_name='revenue')
df['total_year'] = df.groupby(['product', 'region'])['revenue'].transform("sum").copy()

df = df[df['revenue'] > (0.35 * df['total_year'])]
result = pd.pivot_table(df, index='product', columns='quarter', values='revenue', aggfunc='sum')
result = result.fillna(0)

result.to_csv(str(OUT / 'high_impact_quarters.csv'), index=False)

📊 ЗАДАЧА 3: «Когортный анализ удержания клиентов (реальный сценарий)»
#### Контекст: Ты получил лог транзакций из CRM. Данные содержат повторные покупки, записи с отменами (отрицательная сумма) и неравномерную активность. Бизнесу нужно понять, как меняется доля активных пользователей в первые 6 месяцев после первой покупки, чтобы оптимизировать онбординг и коммуникации.

### ТЗ:
     1. Загрузи transactions.csv. Приведи user_id к str, распарси transaction_date
    2. Очисти данные: удали строки, где amount <= 0 (отмены/возвраты не учитываются в метриках удержания). 
    3. Определи когорту для каждого пользователя: месяц его самой первой покупки. Сохрани в колонке cohort в формате 'YYYY-MM'.  
    4. Определи месяц каждой транзакции: сохрани в колонке tx_month в формате 'YYYY-MM'.
    5. Агрегируй данные: сгруппируй по ['cohort', 'tx_month'] и посчитай active_users (число уникальных user_id, совершивших покупку в этот месяц). Сбрось индекс.
    6. Рассчитай размер каждой когорты (cohort_size) и удержание (retention_rate = active_users / cohort_size). Размер когорты равен количеству активных пользователей в её первый месяц.
    7. Отфильтруй результат: оставь только когорты с cohort_size >= 20 и только первые 6 месяцев наблюдения. Разницу в месяцах вычисли как: (год_транзакции - год_когорты) * 12 + (месяц_транзакции - месяц_когорты). Отбери строки, где разница ∈ [0, 5].-->
    8. Сохрани итог в cohort_retention_real.csv без индекса.

✅ Разрешено: read_csv (dtype, parse_dates), pd.to_datetime, dt.year / dt.month / dt.strftime, groupby + transform / agg + nunique, reset_index, sort_values, булева фильтрация, арифметика колонок, to_csv.
❌ Запрещено: merge / concat

📝 Ожидаемый результат: DataFrame с колонками ['cohort', 'tx_month', 'active_users', 'cohort_size', 'retention_rate']. Для каждой когорты первый месяц даёт retention_rate == 1.0. Все значения retention_rate ∈ [0.0, 1.0]. Файл открывается в Excel, готов к загрузке в BI.

In [ ]:
import pandas as pd, numpy as np
## 1. Загрузи transactions.csv. Приведи user_id к str, распарси transaction_date
df = pd.read_csv(str(RAW / 'transactions.csv'), dtype={'user_id': 'str', 'amount': 'float64'}, parse_dates=['transaction_date'])

## 2. Очисти данные: удали строки, где amount <= 0 (отмены/возвраты не учитываются в метриках удержания). 
df = df[df['amount'] > 0]

## 3. Определи когорту для каждого пользователя: месяц его самой первой покупки. Сохрани в колонке cohort в формате 'YYYY-MM'.  
df['cohort'] = (df.groupby('user_id')['transaction_date'].transform('min')).dt.strftime("%Y-%m")

## 4. Определи месяц каждой транзакции: сохрани в колонке tx_month в формате 'YYYY-MM'.
df['tx_month'] = df['transaction_date'].dt.strftime('%Y-%m')

## 5. Агрегируй данные: сгруппируй по ['cohort', 'tx_month'] и посчитай active_users (число уникальных user_id, совершивших покупку в этот месяц). Сбрось индекс.
grp = df.groupby(['cohort', 'tx_month']).agg(active_users=('user_id', 'nunique')).reset_index()

## 6. Рассчитай размер каждой когорты (cohort_size) и удержание (retention_rate = active_users / cohort_size). Размер когорты равен количеству активных пользователей в её первый месяц.
grp['cohort_size'] =  (grp.groupby('cohort')['active_users'].transform('first'))
grp['retention_rate'] = grp['active_users'] / grp['cohort_size']

## 7. Отфильтруй результат: оставь только когорты с cohort_size >= 20 и только первые 6 месяцев наблюдения. 
    ## Разницу в месяцах вычисли как: (год_транзакции - год_когорты) * 12 + (месяц_транзакции - месяц_когорты)
    ## Отбери строки, где разница ∈ [0, 5]
grp['cohort'] = pd.to_datetime(grp['cohort'])
grp['tx_month'] = pd.to_datetime(grp['tx_month'])
result = grp[(grp['cohort_size'] >= 20) & ((((grp['tx_month'].dt.year - grp['cohort'].dt.year) * 12) + (grp['tx_month'].dt.month - grp['cohort'].dt.month)).between(0, 5))].copy()

## 8. Сохрани итог в cohort_retention_real.csv без индекса.
result.to_csv(str(OUT / 'cohort_retention_real.csv'), index=False)

Контекст: Файл raw_events.csv содержит: event_id, user_id, event_date (str, DD.MM.YYYY), event_type, value (float с пропусками).
##### ТЗ:
1. Загрузи с явным dtype (user_id=str, value=float32), распарси дату.
2. Удали дубликаты event_id, оставив первую запись.
3. Замени пропуски в value на медиану по event_type.
4. Отфильтруй события только за 2023 год.
5. Посчитай для каждого event_type: count_events, avg_value, total_value.
6. Добавь колонку share_of_total: доля total_value типа от общей суммы.
7. Сохрани в event_metrics_2023.csv без индекса, с кодировкой utf-8-sig.

##### ✅ Разрешено: read_csv, to_datetime, drop_duplicates, fillna+median, groupby+agg, арифметика колонок, to_csv.
##### ❌ Запрещено: merge, pivot, apply, циклы, ручные расчёты.
##### 📝 Критерий: 0 предупреждений, файл открывается в Excel, share_of_total.sum() ≈ 1.0.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv(str(RAW / 'raw_events.csv'), dtype={'user_id': 'str','value': 'float32'}, parse_dates=['event_date'], date_format='%d.%m.%Y').drop_duplicates(subset=['event_id'])
df['value'] = df['value'].fillna(df.groupby('event_type')['value'].transform('median'))
df = df[df['event_date'].dt.year == 2023]
clmn_result = df.groupby('event_type').agg(count_events=('event_id','count'), avg_value=('value', 'mean'), total_value=('value', 'sum'))
clmn_result['share_of_total'] = clmn_result['total_value'] / clmn_result['total_value'].sum() 
clmn_result.to_csv(str(OUT / 'event_metrics_2023.csv'), index=False, encoding='utf-8-sig')

Контекст: Два файла: users.csv (user_id, region, signup_date) и sessions.csv (session_id, user_id, session_date, duration_sec).

##### ТЗ:
1. Загрузи оба файла с контролем типов. Приведи даты к datetime.
2. В sessions: удали сессии с duration_sec <= 0, замени пропуски в duration_sec на 0.
3. Объедини таблицы через left merge (все пользователи + их сессии). Проверь целостность ключей.
4. Добавь колонку user_tenure_days: разница между session_date и signup_date в днях.
5. Посчитай для каждого region: 
   - total_sessions (число сессий)
   - avg_duration (средняя длительность)
   - active_users (уникальные user_id с duration > 0)
6. Отфильтруй регионы, где active_users >= 30.
7. Сохрани в regional_activity.csv.

##### ✅ Разрешено: read_csv, to_datetime, merge+validate, fillna, groupby+agg+nunique, арифметика дат (.dt.days), булева фильтрация.
##### ❌ Запрещено: apply, циклы, pivot, ручное вычисление разницы дат.
##### 📝 Критерий: После merge количество строк = исходные пользователи × (сессии или 1), если сессий нет.

In [ ]:
import pandas as pd, numpy as np

users = pd.read_csv(str(RAW / 'users.csv'), dtype={'user_id': 'str', 'region': 'str'})
sessions = pd.read_csv(str(RAW / 'sessions.csv'), dtype={'session_id': 'str', 'user_id': 'str'})
users['signup_date'] = pd.to_datetime(users['signup_date'], format='%d.%m.%Y')
sessions['session_date'] = pd.to_datetime(sessions['session_date'], format='%d.%m.%Y')

sessions = sessions[sessions['duration_sec'] >= 0]
sessions['duration_sec'] = sessions['duration_sec'].fillna(0)

users.duplicated(subset=['user_id'], keep='first').sum()

df = pd.merge(users, sessions, on='user_id', how='left', validate='1:m')

df['user_tenure_days'] = (df['session_date'] - df['signup_date']).dt.days
df = df[df['duration_sec'] > 0]

regions_results = df.groupby('region').agg(total_sessions=('session_id','count'), avg_duration=('duration_sec','mean'), active_users=('user_id','nunique'))

results = regions_results[regions_results['active_users'] >= 30]

results.to_csv(str(OUT / 'regional_activity.csv'))



##### Контекст: transactions.csv (user_id, transaction_date, amount, category).

##### ТЗ:
1. Загрузи, очисти: amount > 0, удали дубли (user_id + transaction_date).
2. Определи когорту: месяц первой покупки пользователя (формат 'YYYY-MM').
3. Определи месяц транзакции (формат 'YYYY-MM').
4. Агрегируй: по ['cohort', 'tx_month'] посчитай revenue (sum amount) и buyers (nunique user_id).
5. Рассчитай: 
   - cohort_revenue = revenue в месяце 0 для каждой когорты (используй transform+sort)
   - revenue_retention = revenue / cohort_revenue
6. Разверни через pivot_table: index='cohort', columns='month_diff' (0..5), values='revenue_retention', aggfunc='first', fill_value=0.
7. Сохрани в cohort_dashboard.csv.

##### ✅ Разрешено: Всё из предыдущих + pivot_table, расчёт month_diff через .dt.year/.dt.month.
##### ❌ Запрещено: merge с внешними таблицами, apply, циклы, DateOffset.
##### 📝 Критерий: В колонке month_diff=0 все значения = 1.0. Размер итоговой таблицы: число когорт × 6.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv(str(RAW / 'transactions.csv'), parse_dates=['transaction_date'])
df = df[df['amount'] > 0]
df = df.drop_duplicates(subset=['user_id', 'transaction_date'])
df['cohort'] = df.groupby('user_id')['transaction_date'].transform('first').dt.strftime("%Y-%m")
df['tx_month'] = df['transaction_date'].dt.strftime('%Y-%m')
aggregate = df.groupby(['cohort', 'tx_month']).agg(revenue=('amount', 'sum'), buyers=('user_id', 'nunique')).reset_index() 
aggregate = aggregate.sort_values(['cohort','tx_month'])
aggregate['cohort_revenue'] = aggregate.groupby('cohort')['revenue'].transform('first')
aggregate['revenue_retention'] = aggregate['revenue'] / aggregate['cohort_revenue']
aggregate['tx_month'] = pd.to_datetime(aggregate['tx_month'], format='%Y-%m')
aggregate['cohort'] = pd.to_datetime(aggregate['cohort'], format='%Y-%m')
aggregate['month_diff'] = ((aggregate['tx_month'].dt.year * 12 + aggregate['tx_month'].dt.month) - (aggregate['cohort'].dt.year * 12 + aggregate['cohort'].dt.month))
result = pd.pivot_table(aggregate, index='cohort', columns='month_diff', values='revenue_retention', aggfunc='first', fill_value=0)

result.to_csv(str(OUT / 'cohort_dashboard.csv'))